In [1]:
# 简单通用可视化（Jupyter 单元格）
from pathlib import Path
import pickle, math
import matplotlib.pyplot as plt

# 可选依赖按需导入（缺谁就 None）
try: import pandas as pd
except: pd = None
try: import numpy as np
except: np = None
try: import torch
except: torch = None

def viz_pkl(path):
    p = Path(path)
    with open(p, "rb") as f:
        obj = pickle.load(f)
    print(f"Loaded: {type(obj)}")

    # 1) pandas.DataFrame
    if pd is not None and isinstance(obj, pd.DataFrame):
        display(obj.head(10))
        num_cols = obj.select_dtypes(include="number").columns.tolist()
        if num_cols:
            n = len(num_cols); cols = min(3, n); rows = math.ceil(n/cols)
            plt.figure(figsize=(4*cols, 3*rows))
            for i, c in enumerate(num_cols, 1):
                plt.subplot(rows, cols, i); obj[c].dropna().hist(bins=30); plt.title(c)
            plt.tight_layout(); plt.show()
        return obj

    # 2) numpy.ndarray
    if np is not None and isinstance(obj, np.ndarray):
        print("shape:", obj.shape, "dtype:", obj.dtype)
        if obj.ndim == 1:
            plt.figure(figsize=(6,3)); plt.plot(obj); plt.title("1D array"); plt.show()
        elif obj.ndim == 2:
            plt.figure(figsize=(5,4)); plt.imshow(obj); plt.title("2D array"); plt.colorbar(); plt.show()
        return obj

    # 3) PyTorch: state_dict 或含 state_dict 的对象
    if torch is not None and (
        (isinstance(obj, dict) and any(hasattr(v, "shape") for v in obj.values()))
        or hasattr(obj, "state_dict")
    ):
        sd = obj if isinstance(obj, dict) else obj.state_dict()
        total = 0
        rows = []
        for k, v in sd.items():
            n = int(v.numel()) if hasattr(v, "numel") else 0
            total += n
            rows.append((k, n))
        print(f"Total params: {total:,}")
        rows.sort(key=lambda x: x[1], reverse=True)
        top = rows[:20]
        if top:
            plt.figure(figsize=(8, max(3, len(top)*0.3)))
            plt.barh([k for k,_ in top][::-1], [n for _,n in top][::-1])
            plt.title("Top parameter counts"); plt.tight_layout(); plt.show()
        return obj

    # 4) dict / list：尝试转 DataFrame，否则打印摘要
    if isinstance(obj, (dict, list)):
        if pd is not None:
            try:
                df = pd.DataFrame(obj)
                display(df.head(10))
                return obj
            except Exception:
                pass
        # 摘要打印
        if isinstance(obj, dict):
            print("dict keys (up to 30):", list(obj.keys())[:30])
        else:
            print("list length:", len(obj), "; first 5 types:", [type(x).__name__ for x in obj[:5]])
        return obj

    # 5) 其他类型：打印 repr 以便你后续决定怎么画
    print(repr(obj)[:1000])
    return obj

# 用法：把路径换成你的文件
# obj = viz_pkl("your_file.pkl")


In [3]:
obj = viz_pkl("finetune_data/protein_extraction_10.0A_index.pkl")

Loaded: <class 'list'>


,0,1
0,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
1,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
2,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
3,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
4,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
5,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
6,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
7,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
8,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
9,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...


In [4]:
obj
print(len(obj))

689796


In [5]:
obj[-1]

('/home/yang2531/Documents/Project/Structure_safe/finetune_data/low/target_CHEMBL5648/CHEMBL3644489/protein_10.0A.pdb',
 '/home/yang2531/Documents/Project/Structure_safe/finetune_data/low/target_CHEMBL5648/CHEMBL3644489/ligand.sdf')

In [6]:
obj[300000]

('/home/yang2531/Documents/Project/Structure_safe/finetune_data/high/target_CHEMBL4409/CHEMBL1642582/protein_10.0A.pdb',
 '/home/yang2531/Documents/Project/Structure_safe/finetune_data/high/target_CHEMBL4409/CHEMBL1642582/ligand.sdf')

In [2]:
obj_train = viz_pkl("finetune_data/training_set.pkl")
print(len(obj_train))
obj_train

Loaded: <class 'list'>


,0,1
0,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
1,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
2,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
3,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
4,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
5,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
6,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
7,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
8,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
9,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...


681700


[('/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL2815/CHEMBL369490/protein_10.0A.pdb',
  '/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL2815/CHEMBL369490/ligand.sdf'),
 ('/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL2815/CHEMBL1276446/protein_10.0A.pdb',
  '/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL2815/CHEMBL1276446/ligand.sdf'),
 ('/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL4550/CHEMBL3780838/protein_10.0A.pdb',
  '/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL4550/CHEMBL3780838/ligand.sdf'),
 ('/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL224/CHEMBL1774086/protein_10.0A.pdb',
  '/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL224/CHEMBL1774086/ligand.sdf'),
 ('/home/yang2531/Do

In [5]:
obj_test = viz_pkl("finetune_data/validation_set.pkl")
print(len(obj_test))
obj_test

Loaded: <class 'list'>


,0,1
0,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
1,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
2,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
3,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
4,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
5,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
6,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
7,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
8,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...
9,/home/yang2531/Documents/Project/Structure_saf...,/home/yang2531/Documents/Project/Structure_saf...


228


[('/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL3137292/CHEMBL3099405/protein_10.0A.pdb',
  '/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL3137292/CHEMBL3099405/ligand.sdf'),
 ('/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL1930/CHEMBL4594145/protein_10.0A.pdb',
  '/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL1930/CHEMBL4594145/ligand.sdf'),
 ('/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL1926488/CHEMBL1926835/protein_10.0A.pdb',
  '/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL1926488/CHEMBL1926835/ligand.sdf'),
 ('/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL1075320/CHEMBL1084708/protein_10.0A.pdb',
  '/home/yang2531/Documents/Project/Structure_safe/finetune_data/moderate/target_CHEMBL1075320/CHEMBL1084708/ligand.sdf')

In [1]:
import torch

In [2]:
data = torch.load("finetune_data/processed_data/hf_bdnv2_dataset/ligand_stats.pt")
print(data)

{'mu': tensor([ 9.6470,  4.4209, -7.5277,  ..., 13.5078, 14.0191,  8.4731]), 'sigma': tensor([2.0966, 2.9303, 3.3899,  ..., 6.8708, 4.9199, 4.6296]), 'shape': torch.Size([689796, 1536]), 'num_samples': 689796, 'num_dims': 1536}


/tmp/ipykernel_108311/2760732191.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load("finetune_data/processed_data/hf_bdnv2_dataset/ligand_stats.pt")


In [7]:
print(data['mu'].shape)
data['sigma'].shape

torch.Size([1536])


torch.Size([1536])